# Python Data Pipeline Engineering Lab
## Omnichannel Retail Data Warehouse ETL Pipeline

This notebook demonstrates an incremental, idempotent ETL Pipeline built with Python, Pandas, and SQLite.

### Pipeline Objectives
- **Extract**: Multi-batch order data and dimension sheets from Excel
- **Transform & Data Quality**: Parse dates, validate foreign keys, enforce numerical boundaries, normalize categories, and calculate derived metrics (`gross_amount`, `net_amount`)
- **Quarantine**: Isolate defective records into a Quarantine database table and CSV with explicit reason codes
- **Star Schema Load**: Load normalized data into SQLite Data Warehouse (`dim_customer`, `dim_product`, `dim_date`, `fact_sales`)
- **Idempotency & Watermarking**: Upsert records using `updated_at` timestamps to ensure repeated batch runs yield zero duplicate entries

In [ ]:
import os
import sys
import sqlite3
import pandas as pd
import numpy as np
from pipeline import PipelineConfig, ETLPipeline, run_acceptance_tests

config = PipelineConfig()
print('Pipeline configuration initialized:')
print(f'- Dataset: {config.dataset_path}')
print(f'- Database: {config.db_path}')
print(f'- Quarantine CSV: {config.quarantine_csv_path}')
print(f'- Run Log CSV: {config.run_log_csv_path}')

## Step 1: Run Acceptance Test Suite
Executes 4 sequential runs (`batch_1`, `batch_1 rerun`, `batch_2`, `batch_3`) and validates all acceptance tests.

In [ ]:
run_acceptance_tests(config)

## Step 2: Inspect Star Schema Data Warehouse Tables

In [ ]:
conn = sqlite3.connect(config.db_path)

print('=== FACT_SALES SAMPLE (5 rows) ===')
display(pd.read_sql_query('SELECT * FROM fact_sales LIMIT 5', conn))

print('\n=== DIM_CUSTOMER SAMPLE (5 rows) ===')
display(pd.read_sql_query('SELECT * FROM dim_customer LIMIT 5', conn))

print('\n=== DIM_PRODUCT SAMPLE (5 rows) ===')
display(pd.read_sql_query('SELECT * FROM dim_product LIMIT 5', conn))

conn.close()

## Step 3: Pipeline Execution Log & Quarantine Analysis

In [ ]:
log_df = pd.read_csv(config.run_log_csv_path)
print('=== PIPELINE RUN LOG ===')
display(log_df)

q_df = pd.read_csv(config.quarantine_csv_path)
print(f'\n=== QUARANTINE RECORDS SUMMARY ({len(q_df)} total records) ===')
print(q_df['reason_code'].value_counts())
display(q_df.head(10))